## Step 1: Problem Statement

You have customer data:

Plain Text - CreditScore
Age
Balance
Salary
Geography
Gender

Target column: Exited
where - 0 = Customer stayed
1 = Customer left


Given a customer's information, predict whether they are likely to leave the bank.

In [73]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,LabelEncoder
import pickle

In [74]:
data = pd.read_csv("Churn_Modelling.csv")
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


## Step 2: Feature Transformation
Raw data can't be directly fed into a neural network.
Neural networks understand only numbers.
So you performed:
Label Encoding 
One Hot Encoding

In [75]:
##Preprocessing the data
data =data.drop(['RowNumber','CustomerId','Surname'],axis=1)
data

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0
...,...,...,...,...,...,...,...,...,...,...,...
9995,771,France,Male,39,5,0.00,2,1,0,96270.64,0
9996,516,France,Male,35,10,57369.61,1,1,1,101699.77,0
9997,709,France,Female,36,7,0.00,1,0,1,42085.58,1
9998,772,Germany,Male,42,3,75075.31,2,1,0,92888.52,1


In [76]:
##encode categorical data --> 0 and 1 for gender
label_encoder_gender =LabelEncoder()
data['Gender'] =label_encoder_gender.fit_transform(data['Gender'])
data

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,0,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,0,41,1,83807.86,1,0,1,112542.58,0
2,502,France,0,42,8,159660.80,3,1,0,113931.57,1
3,699,France,0,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,0,43,2,125510.82,1,1,1,79084.10,0
...,...,...,...,...,...,...,...,...,...,...,...
9995,771,France,1,39,5,0.00,2,1,0,96270.64,0
9996,516,France,1,35,10,57369.61,1,1,1,101699.77,0
9997,709,France,0,36,7,0.00,1,0,1,42085.58,1
9998,772,Germany,1,42,3,75075.31,2,1,0,92888.52,1


In [77]:
## Onehot encoding the geography column because using 0,1,2 will make the model think that 2 is greater than 1 and 1 is greater than 0 which is not true. So we will use one hot encoding to create dummy variables for each category in the geography column.
from sklearn.preprocessing import OneHotEncoder
onehot_encoder_geography = OneHotEncoder()
geo_encoder = onehot_encoder_geography.fit_transform(data[['Geography']])
geo_encoder

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 10000 stored elements and shape (10000, 3)>

In [78]:
geo_encoder.toarray()

array([[1., 0., 0.],
       [0., 0., 1.],
       [1., 0., 0.],
       ...,
       [1., 0., 0.],
       [0., 1., 0.],
       [1., 0., 0.]], shape=(10000, 3))

In [79]:
onehot_encoder_geography.get_feature_names_out(['Geography'])

array(['Geography_France', 'Geography_Germany', 'Geography_Spain'],
      dtype=object)

In [80]:
geo_encoded_df=pd.DataFrame(geo_encoder.toarray(), columns=onehot_encoder_geography.get_feature_names_out(['Geography']))

In [81]:
geo_encoded_df

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,0.0,0.0,1.0
...,...,...,...
9995,1.0,0.0,0.0
9996,1.0,0.0,0.0
9997,1.0,0.0,0.0
9998,0.0,1.0,0.0


In [82]:
## combine one hot encoder columns with the original data
data =pd.concat([data.drop(['Geography'],axis=1),geo_encoded_df],axis=1 )

In [83]:
data.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [84]:
## save the encoders and scalar

with open('label_encoder_gender.pkl', 'wb') as file:
    pickle.dump(label_encoder_gender, file)

with open('onehot_encoder_geography.pkl', 'wb') as file:
    pickle.dump(onehot_encoder_geography, file)



## Step 3: Train-Test Split
10000 records


8000 → Training
2000 → Testing

In [85]:
## divide the data into ondependent and dependent variables
X = data.drop(['Exited'],axis=1)
Y=data['Exited']

## split the data into training and testing data
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

##Scale the data using standard scaler
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)



In [86]:
X_train

array([[ 0.35649971,  0.91324755, -0.6557859 , ...,  1.00150113,
        -0.57946723, -0.57638802],
       [-0.20389777,  0.91324755,  0.29493847, ..., -0.99850112,
         1.72572313, -0.57638802],
       [-0.96147213,  0.91324755, -1.41636539, ..., -0.99850112,
        -0.57946723,  1.73494238],
       ...,
       [ 0.86500853, -1.09499335, -0.08535128, ...,  1.00150113,
        -0.57946723, -0.57638802],
       [ 0.15932282,  0.91324755,  0.3900109 , ...,  1.00150113,
        -0.57946723, -0.57638802],
       [ 0.47065475,  0.91324755,  1.15059039, ..., -0.99850112,
         1.72572313, -0.57638802]], shape=(8000, 12))

In [87]:
X_train.shape, X_test.shape, Y_train.shape, Y_test.shape

((8000, 12), (2000, 12), (8000,), (2000,))

In [88]:
with open('scalar.pkl', 'wb') as file:
    pickle.dump(scaler, file)

# ANN Implementation


In [89]:
import sys
print(sys.executable)

c:\Users\nehnayak\OneDrive - Capgemini\Desktop\Generative AI\venv\Scripts\python.exe


In [90]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense 
from tensorflow.keras.callbacks import EarlyStopping,TensorBoard 
import datetime

In [91]:
(X_train.shape[1],)

(12,)

In [92]:
##Build an ANN model using Keras Sequential API

model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train.shape[1],)), ## First hidden layer with 64 neurons and relu activation function
    Dense(32, activation='relu'),## Hidden layer with 32 neurons and relu activation function
    Dense(1, activation='sigmoid') ## Output layer with 1 neuron and sigmoid activation function
])


c:\Users\nehnayak\OneDrive - Capgemini\Desktop\Generative AI\venv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [93]:
model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_8 (Dense)                 │ (None, 64)             │           832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,945 (11.50 KB)

 Trainable params: 2,945 (11.50 KB)

 Non-trainable params: 0 (0.00 B)

In [94]:
import tensorflow
opt= tensorflow.keras.optimizers.Adam(learning_rate=0.001)
loss=tensorflow.keras.losses.BinaryCrossentropy()
loss

<LossFunctionWrapper(<function binary_crossentropy at 0x0000021CD3CDB600>, kwargs={'from_logits': False, 'label_smoothing': 0.0, 'axis': -1})>

In [95]:
## compile the model
model.compile(optimizer="adam",loss="binary_crossentropy",metrics=["accuracy"])

## 1. What is TensorBoard?

TensorBoard is TensorFlow's visualization tool. It helps you monitor:

Training loss
Validation loss
Accuracy
Model graph
Weights and biases
Histograms of activations

Instead of just seeing numbers in the console, you get interactive graphs.

In [96]:
#Set up the Tensorboard

from tensorflow.keras.callbacks import TensorBoard,EarlyStopping  

log_dir = "logs/fit" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorflow_callback =  TensorBoard(log_dir = log_dir,histogram_freq=1)

## Early Stopping
Early Stopping is a Keras callback that monitors a metric such as validation loss and automatically stops training when performance stops improving, helping prevent overfitting and reducing unnecessary computation

In [99]:
## set up early stopping to prevent overfitting

early_stopping_callback = EarlyStopping(monitor='val_loss',patience=10,restore_best_weights=True)

## POINTS TO REMEMBER
 patience=10 - Wait 10 epochs for improvement


## Why restore_best_weights=True?

Even though training stopped at Epoch 12, TensorFlow reloads the weights from Epoch 2 because that had the lowest validation loss.

So your final model is effectively the Epoch 2 model, not Epoch 12.

## Training accuracy vs Validation Accuracy

Case - 1

Training Accuracy   = 92%
Validation Accuracy = 90%
92 - 90 = 2%
This is good.
The model learned meaningful patterns that also work on unseen data.

Case - 2
Training Accuracy   = 99%
Validation Accuracy = 82%
99 - 82 = 17%
The model performs extremely well on training data but poorly on unseen data.

Training accuracy measures performance on data used for learning, while validation accuracy measures performance on unseen data. If training accuracy keeps increasing but validation accuracy stops improving or decreases, the model is overfitting. It means the network is memorizing training examples instead of learning general patterns that can be applied to new data.

In [100]:
### train the model
history = model.fit(
    X_train,Y_train,validation_data=(X_test,Y_test),epochs=100,
    callbacks=[tensorflow_callback,early_stopping_callback]
)

Epoch 1/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8666 - loss: 0.3182 - val_accuracy: 0.8615 - val_loss: 0.3427
Epoch 2/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8685 - loss: 0.3177 - val_accuracy: 0.8660 - val_loss: 0.3392
Epoch 3/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8673 - loss: 0.3160 - val_accuracy: 0.8635 - val_loss: 0.3432
Epoch 4/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8700 - loss: 0.3151 - val_accuracy: 0.8640 - val_loss: 0.3419
Epoch 5/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.8712 - loss: 0.3139 - val_accuracy: 0.8600 - val_loss: 0.3464
Epoch 6/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.8687 - loss: 0.3124 - val_accuracy: 0.8605 - val_loss: 0.3406
Epoch 7/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.8677 - loss: 0.3118 - val_accuracy: 0.8570 - val_loss: 0.3457
Epoch 8/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.8719 - loss: 0.3104 - val_accu

In [101]:
%load_ext tensorboard

In [103]:
%tensorboard --logdir logs/fit

Reusing TensorBoard on port 6006 (pid 29280), started 0:02:40 ago. (Use '!kill 29280' to kill it.)

Tensorflow Dashboard link - http://localhost:6007/

tensorboard --logdir logs                                                                                                            

In [104]:


model.save('model.h5')